The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

In [1]:
!pip install git+https://github.com/d2l-ai/d2l-zh@release  # installing d2l


  Cloning https://github.com/d2l-ai/d2l-zh (to revision release) to /tmp/pip-req-build-3w2r99ar
  Running command git clone --filter=blob:none --quiet https://github.com/d2l-ai/d2l-zh /tmp/pip-req-build-3w2r99ar
  Running command git checkout -b release --track origin/release
  Switched to a new branch 'release'
  Branch 'release' set up to track remote branch 'release' from 'origin'.
  Resolved https://github.com/d2l-ai/d2l-zh to commit 843d3d41dca48d8df65f4b324dd171d8bfe9c067
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of d2l to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 

# 多层感知机的从零开始实现
:label:`sec_mlp_scratch`

我们已经在 :numref:`sec_mlp`中描述了多层感知机（MLP），
现在让我们尝试自己实现一个多层感知机。
为了与之前softmax回归（ :numref:`sec_softmax_scratch` ）
获得的结果进行比较，
我们将继续使用Fashion-MNIST图像分类数据集
（ :numref:`sec_fashion_mnist`）。


In [2]:
import torch
from torch import nn
from d2l import torch as d2l

ModuleNotFoundError: No module named 'd2l'

In [3]:
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

NameError: name 'd2l' is not defined

## 初始化模型参数

回想一下，Fashion-MNIST中的每个图像由
$28 \times 28 = 784$个灰度像素值组成。
所有图像共分为10个类别。
忽略像素之间的空间结构，
我们可以将每个图像视为具有784个输入特征
和10个类的简单分类数据集。
首先，我们将[**实现一个具有单隐藏层的多层感知机，
它包含256个隐藏单元**]。
注意，我们可以将这两个变量都视为超参数。
通常，我们选择2的若干次幂作为层的宽度。
因为内存在硬件中的分配和寻址方式，这么做往往可以在计算上更高效。

我们用几个张量来表示我们的参数。
注意，对于每一层我们都要记录一个权重矩阵和一个偏置向量。
跟以前一样，我们要为损失关于这些参数的梯度分配内存。


In [4]:
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

## 激活函数

为了确保我们对模型的细节了如指掌，
我们将[**实现ReLU激活函数**]，
而不是直接调用内置的`relu`函数。


In [5]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

## 模型

因为我们忽略了空间结构，
所以我们使用`reshape`将每个二维图像转换为一个长度为`num_inputs`的向量。
只需几行代码就可以(**实现我们的模型**)。


In [6]:
def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(X@W1 + b1)  # 这里“@”代表矩阵乘法
    return (H@W2 + b2)

## 损失函数

由于我们已经从零实现过softmax函数（ :numref:`sec_softmax_scratch`），
因此在这里我们直接使用高级API中的内置函数来计算softmax和交叉熵损失。
回想一下我们之前在 :numref:`subsec_softmax-implementation-revisited`中
对这些复杂问题的讨论。
我们鼓励感兴趣的读者查看损失函数的源代码，以加深对实现细节的了解。


In [7]:
loss = nn.CrossEntropyLoss(reduction='none')

## 训练

幸运的是，[**多层感知机的训练过程与softmax回归的训练过程完全相同**]。
可以直接调用`d2l`包的`train_ch3`函数（参见 :numref:`sec_softmax_scratch` ），
将迭代周期数设置为10，并将学习率设置为0.1.


In [8]:
num_epochs, lr = 10, 0.1
updater = torch.optim.SGD(params, lr=lr)
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, updater)

NameError: name 'd2l' is not defined

为了对学习到的模型进行评估，我们将[**在一些测试数据上应用这个模型**]。


In [9]:
d2l.predict_ch3(net, test_iter)

NameError: name 'd2l' is not defined

## 小结

* 手动实现一个简单的多层感知机是很容易的。然而如果有大量的层，从零开始实现多层感知机会变得很麻烦（例如，要命名和记录模型的参数）。

## 练习

1. 在所有其他参数保持不变的情况下，更改超参数`num_hiddens`的值，并查看此超参数的变化对结果有何影响。确定此超参数的最佳值。
1. 尝试添加更多的隐藏层，并查看它对结果有何影响。
1. 改变学习速率会如何影响结果？保持模型架构和其他超参数（包括轮数）不变，学习率设置为多少会带来最好的结果？
1. 通过对所有超参数（学习率、轮数、隐藏层数、每层的隐藏单元数）进行联合优化，可以得到的最佳结果是什么？
1. 描述为什么涉及多个超参数更具挑战性。
1. 如果想要构建多个超参数的搜索方法，请想出一个聪明的策略。


1. 调节隐藏单元数 num_hiddens 的影响与选择影响机理：过小（如 16, 32）： 模型容量（Capacity）不足，无法充分学习 Fashion-MNIST 的非线性特征，容易发生欠拟合，训练集和测试集准确率均偏低。适中（如 128, 256）： 模型表达能力与任务复杂度匹配良好，收敛平稳，泛化表现最佳。过大（如 512, 1024）： 参数量显著增加，计算耗时上升，且在没有正则化（如 Dropout、权重衰减）的情况下容易导致过拟合（训练损失极低但测试集精度不再上升甚至回落）。最佳值推荐： 在保持原代码（无 Dropout/Weight Decay、10 轮迭代）不变时，通常 num_hiddens = 256 或 128 是最佳折中点，测试集准确率通常能稳定在 84%~85% 左右。

2. 添加更多隐藏层的影响实验表现： 增加到 2~3 层隐藏层（例如 784 -> 256 -> 128 -> 10）时，通常会发现效果并没有明显提升，甚至可能下降。原因分析：架构与任务不匹配： Fashion-MNIST 仅为 28×28 单通道灰度图，单层宽隐层（如 256）的拟合能力已足以表达该任务的特征分布。优化困难： 网络加深后，简单的固定小方差随机初始化（randn * 0.01）与基础的 SGD 容易导致梯度消失或信号弥散，在仅训练 10 轮的情况下往往尚未充分收敛。深层网络通常需要配合批量归一化（BatchNorm）、残差连接或更成熟的初始化（如 He 初始化）。

3. 改变学习率（Learning Rate, lr）的影响与调优学习率区间实验现象背后原因过大（如 lr = 1.0 或更高）损失震荡不降，甚至变为 nan，准确率停留在 10% 左右（随机猜测）参数更新步长过大，直接越过局部极小点发生梯度爆炸或发散过小（如 lr = 0.001）训练损失下降极其缓慢，10 轮后准确率仅有 60%~70%步长太小，在固定的 10 轮内根本没有走到收敛区域最佳区间（0.1 ~ 0.5）损失平滑稳定下降，10 轮内收敛到较优精度步长合理，能平衡探索速度与收敛精度最佳设置： 保持原代码其他配置时，lr = 0.2 ~ 0.3 往往收敛更快，在 10 轮结束时取得最好的测试准确率（约 85% 上下）。

4. 联合优化所有超参数可达到的最佳表现在纯 MLP 架构（全连接层）下，联合调节超参数后，Fashion-MNIST 的测试集准确率上限大约在 88% ~ 89.5% 之间（很难突破 90%，因为全连接层缺少 CNN 的平移不变性与局部感受野）：典型最佳配置组合：架构： 2 层隐藏层（如 [384, 128]）并加入 Dropout(0.2) 防止过拟合。学习率与优化器： lr = 0.1 配合学习率衰减（如 Cosine Annealing），或直接使用 AdamW(lr=1e-3, weight_decay=1e-4)。迭代轮数： 增加至 25 ~ 40 epochs。激活函数与初始化： 使用 Kaiming 正态初始化，激活函数可选 ReLU 或 GELU。

5. 为什么涉及多个超参数极具挑战？维度灾难（Curse of Dimensionality）： 搜索空间随超参数数量呈指数级爆炸。若有 4 个超参数，每个仅选 5 个候选值，网格搜索需要遍历 $5^4 = 625$ 次完整训练；若每轮训练需要 2 分钟，总耗时达 20 小时以上。参数间存在强耦合（Non-linear Interactions）： 超参数不是独立的。例如：增大批大小（batch size）通常需要同比例放大学习率；网络层数加深通常需要调小学习率或使用更精细的初始化；迭代轮数多少完全取决于学习率大小。单独调节某一参数往往得不到全局最优。评估成本极高： 评估一组超参数的优劣需要跑完一次完整的模型训练与验证，时间成本昂贵，且带有随机种子带来的评估噪声。

6. 高效多超参数搜索策略如果设计一套现代化的超参数搜索系统，推荐采用由粗到细的自适应搜索策略：第一阶段：贝叶斯优化（Bayesian Optimization / TPE）替代网格搜索使用高斯过程（GP）或树状结构 Parzen 估计器（TPE，如开源库 Optuna、Ray Tune）。根据此前尝试过的参数及对应的验证集损失，构建概率代理模型，优先探索“最有希望取得更好结果”的参数区域，搜索效率比随机搜索高数倍。第二阶段：结合早停机制（Successive Halving / ASHA 算法）不要让表现糟糕的试验跑满整个训练轮数。例如训练 3 轮后，淘汰验证集效果排在后 50% 的参数配置；只给表现好的超参数分配算力跑满 20 或 40 轮。这能节省 70% 以上的无效算力。经验对数尺度采样（Log-scale Sampling）：学习率、权重衰减等跨数量级的超参数，必须在对数空间均匀采样（如在 $[10^{-4}, 10^0]$ 上取对数均匀分布），而非线性空间采样。

[Discussions](https://discuss.d2l.ai/t/1804)
